# आपके Google Drive में बनी वास्तविक PDF का हिसाब
यह अलग, सुरक्षित **केवल जाँचने वाला** Notebook है। इससे PDF बनाई या डिलीट नहीं जाएगी। Run करने पर **कुल PDF**, **कुल पेज**, **सबसे ज्यादा पेज वाली PDF**, और **Top 10** स्क्रीन पर मिलेंगे। परिणाम `REPORTS/ACTUAL_PDF_TOTALS.txt` और `ACTUAL_PDF_PAGE_AUDIT.csv` में भी सेव होगा।
वही Drive फ़ोल्डर इस्तेमाल होगा: `MyDrive/BTC_15MIN_1H_2014_2026_V3_RESEARCH/ONE_15MIN_PDF_SET`। अगर PDF अभी नहीं बनी हैं तो यह `RUN_STATUS.txt` / `LAST_ERROR.txt` दिखाएगा।


In [ ]:
import sys, subprocess, csv
from pathlib import Path
from datetime import datetime, timezone

try:
    from pypdf import PdfReader
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pypdf"])
    from pypdf import PdfReader

from google.colab import drive
drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/BTC_15MIN_1H_2014_2026_V3_RESEARCH")
FOLDER = ROOT / "ONE_15MIN_PDF_SET"
REPORT = ROOT / "REPORTS"
REPORT.mkdir(parents=True, exist_ok=True)
CACHE = REPORT / "ACTUAL_PDF_PAGE_AUDIT.csv"
SUMMARY = REPORT / "ACTUAL_PDF_TOTALS.txt"
FIELDNAMES = ["filename", "pages", "bytes", "modified_ns", "status"]

print("Checking actual PDFs in:", FOLDER)
paths = sorted(FOLDER.glob("*.pdf")) if FOLDER.is_dir() else []
print("Actual PDF files currently found:", len(paths))

if not paths:
    print("\nNo PDF files found in this output folder.")
    status = ROOT / "RUN_STATUS.txt"
    err = ROOT / "LAST_ERROR.txt"
    if status.exists():
        print("\nRUN_STATUS.txt:\n", status.read_text(encoding="utf-8", errors="replace")[:3000])
    if err.exists():
        print("\nLAST_ERROR.txt (last lines):\n", err.read_text(encoding="utf-8", errors="replace")[-4500:])
else:
    previous = {}
    if CACHE.exists():
        with CACHE.open(newline="", encoding="utf-8") as fh:
            for record in csv.DictReader(fh):
                previous[record["filename"]] = record

    def save(rows):
        temp = REPORT / "ACTUAL_PDF_PAGE_AUDIT.tmp.csv"
        with temp.open("w", newline="", encoding="utf-8") as fh:
            writer = csv.DictWriter(fh, fieldnames=FIELDNAMES)
            writer.writeheader()
            writer.writerows(rows)
        temp.replace(CACHE)

    results = []
    for index, pdf in enumerate(paths, 1):
        stat = pdf.stat()
        old = previous.get(pdf.name)
        if (old
            and old.get("status") == "OK"
            and old.get("bytes") == str(stat.st_size)
            and old.get("modified_ns") == str(stat.st_mtime_ns)):
            count = int(old["pages"])
            status = "OK"
        else:
            try:
                count = len(PdfReader(str(pdf), strict=False).pages)
                status = "OK"
            except Exception as exc:
                count = 0
                status = type(exc).__name__ + ": " + str(exc)[:150]

        results.append({
            "filename": pdf.name,
            "pages": count,
            "bytes": stat.st_size,
            "modified_ns": stat.st_mtime_ns,
            "status": status
        })

        if index % 50 == 0 or index == len(paths):
            save(results)
            print(f"Checked {index:,} / {len(paths):,} PDFs")

    good = [x for x in results if x["status"] == "OK"]
    bad = [x for x in results if x["status"] != "OK"]
    all_pages = sum(x["pages"] for x in good)
    ordered = sorted(good, key=lambda x: (-x["pages"], x["filename"]))
    largest = ordered[0] if ordered else None

    lines = [
        "ACTUAL PDF COUNT REPORT (FILES PRESENT IN GOOGLE DRIVE)",
        f"UTC checked: {datetime.now(timezone.utc).isoformat()}",
        f"Folder: {FOLDER}",
        f"PDF files present: {len(paths):,}",
        f"PDF files readable: {len(good):,}",
        f"PDF files unreadable: {len(bad):,}",
        f"TOTAL PAGES (readable PDFs): {all_pages:,}",
        f"LARGEST PDF: {largest['filename'] if largest else 'NONE'}",
        f"LARGEST PDF PAGE COUNT: {largest['pages'] if largest else 0:,}",
        "",
        "TOP 10 PDF FILES BY NUMBER OF PAGES:"
    ]
    for rank, row in enumerate(ordered[:10], 1):
        lines.append(f"{rank}. {row['filename']} - {row['pages']:,} pages")
    if bad:
        lines.append("")
        lines.append("UNREADABLE FILES:")
        for row in bad[:30]:
            lines.append(f"{row['filename']}: {row['status']}")
    SUMMARY.write_text("\n".join(lines) + "\n", encoding="utf-8")
    print("\n" + "\n".join(lines))
    print("\nSaved counts:", CACHE)
    print("Saved summary:", SUMMARY)
